# 整体架构：  
$Encoder(负责提取特征)----->Decoder(利用特征进行预测)$

## Encoder

### Attention 注意力机制： 

每一个词或每一个特征点都会与上下文进行权重加和计算，得到一个新的融合了上下文信息的向量

序列中的每一个词或者图中的每一个特征点都会映射成向量 $X_n$  

经由可训练的权重矩阵 $W$ 将$X$,映射成各自的 $Q$, $K$, $V$  $(X, Q, K, V的维度都相同，因为要做残差连接)$ 

$Q$ = $X$ $\times$ $W_q$  
$K$ = $X$ $\times$ $W_k$  
$V$ = $X$ $\times$ $W_v$ 

每次反向传播更新的参数也只有 $W_q$, $W_k$, $W_v$  
而损失则是根据预测结果与实际标签的差距来确定

注意力分数计算：  
$\text Scale Dot-Product Attention$ 计算公式：  
$$
\text{AttentionScore} = \text{softmax}\left(\frac{Q \times K^T}{\sqrt{d_k}}\right)
$$

$d_k$:向量维度，引入向量维度消除维度对于训练稳定性的影响，防止在高纬度下，softmax梯度消失
  
Attention整体计算流程：  
1. 每个词的$Q$都会跟每一个词的$K$计算得分  
2. 得分 / $\sqrt{d_k}$
3. softmax后得到整个加权结果
4. $z_1 = s_1 \times v_1 + s_2 \times v_2 + ..... s_n \times v_n$ 
5. 统一时间计算出所有词的表示结果

### multi-headed 多头机制：

多头的本质是让模型在多个不同的表示子空间里并行地学习不同的注意力模式

$经典平均分割$：
- 同一个向量$X$，经由多个随机初始化的权重矩阵${W_n^q},{W_n^k},{W_n^v}$得到多组$q,k,v$形成多头，得到的注意力结果不同，得到的特征向量表达也不同

例子：  
输入 X 是 512 维向量  
每个头有自己的三个权重矩阵
8 个头并行做注意力，输出各自是 64 维，拼起来回到 512 维，再过一个输出投影。


多头注意力机制计算流程：
1. 通过不同的$head$得到多个特征表达  
2. 将所有特征拼接在一起
3. 在通过一层全连接层进行降维

### 位置信息表达：

Transformer对于位置信息并不敏感  
你打我 vs 我打你  
在Transformer中都有同一组qkv进行表示，但是在实际语义中二者天差地别，所以在Transformer中引入了位置信息的表达

### Transformer由多层堆叠
编码器的一层 = Self-Attention + FC(全连接层)

## Decoder

decoder从目标序列的embedding+位置编码来投影出每个向量qkv  
1. 通过遮罩自注意力只能看到前文的，输出一个融合前文的对V进行加权求和后的新向量
2. 通过交叉注意力来结合encoder提取的特征进行预测，拿着decoder的q去找encoder的输出kv,最后输出encoder各位置v向量的加权和
3. 通过前馈神经网络整理前面两步聚合来的信息

# 手撕Transformer

#### Embedding 模块

In [2]:
import torch
from torch import nn
import torch.nn.functional as F
import math

In [3]:
random_torch = torch.rand(4, 4)
print(random_torch)

tensor([[0.7947, 0.0745, 0.3507, 0.0435],
        [0.4350, 0.0211, 0.2914, 0.6463],
        [0.3038, 0.2871, 0.1215, 0.3933],
        [0.6614, 0.0839, 0.3941, 0.3560]])


In [ ]:
from torch import Tensor

# 将输入的词汇表索引转换为指定维度的Embedding
# 把 token ID 变成向量
class TokenEmbedding(nn.Embedding):
    def __init__(self, vocab_size, d_model):  # vocab_size:词汇表的大小  d_model:模型的维度
        # 在词表构建时约定了ID为1的行嵌入是填充，训练时不更新、梯度不传
        super(TokenEmbedding, self).__init__(vocab_size, d_model, padding_idx=1)

In [ ]:
# 给输入向量注入位置信息
class PositionalEmbedding(nn.Module):
    def __init__(self, d_model, max_len, device):
        super(PositionalEmbedding, self).__init__()
        # 预存的位置编码矩阵，用于存贮最终位置信息
        self.encoding = torch.zeros(max_len, d_model, device=device)  # 设置一个大小为(max_len, d_model)的全零矩阵
        self.encoding.requires_grad = False  # 取消梯度，因为位置矩阵不需要反向传播
        # 位置序号（确定self.encoding的列）
        pos = torch.arange(0, max_len, device=device)
        pos = pos.float().unsqueeze(1) # pos为一维向量(seq_len, ),后面要进行矩阵计算所以要转为二维张量
        # 维度索引的偶数部分（确定self.encoding的行）
        _2i = torch.arange(0, d_model, step=2, device=device).float()  # 生成维度索引序列 2i代表偶数维度的索引值
        '''
        以pos来定义每个输入向量的位置
        通过区分奇数和偶数的索引
        _2i带入到对应的公式来算出一组对应的频率缩放因子
        最终输出每一个pos的多个维度的位置信息
        ''' 
        self.encoding[:, 0::2] = torch.sin(pos/(10000**(_2i/d_model)))  # 偶数维度
        self.encoding[:, 1::2] = torch.cos(pos/(10000**(_2i/d_model)))  # 奇数维度

    def forward(self, x):
        batch_size, seq_len = x.size()
        return self.encoding[:seq_len, :]  # self.encoding是预存好的位置编码矩阵，但序列不一定有这么长，对self.encoding切割到合适长度

In [ ]:
class TransformerEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model, max_len, drop_prob, device):
        super(TransformerEmbedding).__init__()
        self.tok_emb = TokenEmbedding(vocab_size, d_model)
        self.pos_emb = PositionalEmbedding(d_model, max_len, device)
        self.dropout = nn.Dropout(p=drop_prob)  # dropout率

    def forward(self, x):
        tok_emb = self.tok_emb(x)
        pos_emb = self.pos_emb(x)
        return self.dropout(tok_emb+pos_emb)